## create_your_own_probability_distribution

In [6]:
# conda create -n ai_course python=3.11 -y


# conda activate ai_course

In [7]:
conda create -n ai_course python=3.11 -y

Retrieving notices: ...working... DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): repo.anaconda.com:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): repo.anaconda.com:443
DEBUG:urllib3.connectionpool:https://repo.anaconda.com:443 "GET /pkgs/main/notices.json HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://repo.anaconda.com:443 "GET /pkgs/r/notices.json HTTP/1.1" 404 None
done
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): repo.anaconda.com:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): repo.anaconda.com:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): repo.anaconda.com:443
- DEBUG:urllib3.connectionpool:https://repo.anaconda.com:443 "GET /pkgs/r/osx-arm64/current_repodata.json HTTP/1.1" 304 0
DEBUG:urllib3.connectionpool:https://repo.anaconda.com:443 "GET /pkgs/r/noarch/current_repodata.json HTTP/1.1" 304 0
\ DEBUG:urllib3.connectionpool:https://repo.anaconda.com:443 "GET /pkgs/main/noar

In [ ]:
conda update -n base -c defaults conda

Solving environment: unsuccessful attempt using repodata from current_repodata.json, retrying with next repodata source.
Solving environment: unsuccessful attempt using repodata from current_repodata.json, retrying with next repodata source.
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): repo.anaconda.com:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): repo.anaconda.com:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): repo.anaconda.com:443
- DEBUG:urllib3.connectionpool:https://repo.anaconda.com:443 "GET /pkgs/r/osx-arm64/repodata.json HTTP/1.1" 304 0
\ DEBUG:urllib3.connectionpool:https://repo.anaconda.com:443 "GET /pkgs/main/noarch/repodata.json HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://repo.anaconda.com:443 "GET /pkgs/r/noarch/repodata.json HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://repo.anaconda.com:443 "GET /pkgs/main/osx-arm64/repodata.json HTTP/1.1" 200 None
done
Solving environment: 

## Overview

In the previous activities, you have been introduced to language models. Fundamentally, a language model assigns a probability distribution to the next word for a given **context**. In later parts of this course, you will learn how you can build models that automatically compute these probability distributions. But before that, it is important that you get a sense of how these probability distributions are used to predict the next word.

In this lab, you will generate continuations for prompts by randomly choosing words from a probability distribution. A **prompt** is the input text provided to a language model. The goal is to assign probability values to a set of candidate words based on the context defined by the prompt. Then randomly choose from this list, a process known as **sampling**, to generate the next word. You will investigate how probabilities influence the generation of language and practice manipulating these probabilities to create sensible sentences. In doing so, you will mimic the basic principles of language models.

The prompt you will use throughout this course is "Jide was hungry so she went looking for." It is adapted from the prompt featured in Eldan and Li's 2023 paper [_TinyStories: How Small Can Language Models Be and Still Speak Coherent English?_](https://arxiv.org/pdf/2305.07759) [1].




### What you will learn:

By the end of this lab, you will understand:

* How probabilities influence the choice of the next word.
* How the context should determine the probability distribution over the next word.

### Tasks

**In this lab, you will**:
* Assign probabilities to each candidate word reflecting the likelihood of it being the appropriate next word given the prompt.
* Use the `random.choices` function with the assigned probabilities to sample a candidate word.
* Explore how altering the prompt (e.g., changing *hungry* to *thirsty* or modifying the subject) influences the probability distribution and the predicted next word.

All of these steps are described in detail in the following sections.

In [1]:
from datetime import datetime

print(f"Today is {datetime.today():%A}.")

Today is Sunday.


## Imports

In this lab, you will make use of the `random` package to randomly pick elements from a probability distribution. All labs also use functions from the custom `ai_foundations` package that implements specific functionality for this course, such as automatic tests for verifying your answers.

Run the following cell to import the required packages.

In [5]:

# Install the custom package for this course.
!pip install "git+https://github.com/google-deepmind/ai-foundations.git@main"



  Cloning https://github.com/google-deepmind/ai-foundations.git (to revision main) to /private/var/folders/h1/7f4bpj7537vgrwpzrc08_hnm0000gn/T/pip-req-build-qb9563cf
  Running command git clone --filter=blob:none --quiet https://github.com/google-deepmind/ai-foundations.git /private/var/folders/h1/7f4bpj7537vgrwpzrc08_hnm0000gn/T/pip-req-build-qb9563cf
  Resolved https://github.com/google-deepmind/ai-foundations.git to commit 566897a4ec6370e7d136e39e52e02fd0335dec6b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of ai-foundations to determine which version is compatible with other requirements. This could take a while.
ERROR: Ignored the following versions that require a different python version: 2.0.0 Requires-Python >=3.11; 2.0.1 Requires-Python >=3.11; 2.0.2 Requires-Python >=3.11; 2.0.3 Requires-Python >=3.11; 2.0.4 Requires-Python >=3.11; 2.0.5 Requires

In [4]:
# Packages used.
# For randomly picking elements from a probability distribution.
import random
# Custom functions for providing feedback on your solutions.
from ai_foundations.feedback.course_1 import probabilities

ModuleNotFoundError: No module named 'ai_foundations'

Note: All words have been transformed to lowercase and punctuation marks have been split from the words. Assume that punctuation marks are considered to be individual words as well (e.g., “spicy , tomato-based” is split into the three words “spicy”, “,”, and “tomato-based”).
Compute the number of trigrams made up of any combination of words that appear in this text whose count is 0.

In [1]:
text = "jollof rice is a beloved one-pot rice dish from west africa , characterized by its vibrant reddish-orange color and rich , often slightly spicy , tomato-based flavor infused with onions , peppers , garlic , ginger , curry powder , and thyme , all cooked in stock."
words = text.split()
print("Total words:", len(words))
unique_words = set(words)
V = len(unique_words)
print("Unique words (V):", V)

# Compute all trigrams that appear in the text
observed_trigrams = set()
for i in range(len(words) - 2):
    trigram = (words[i], words[i+1], words[i+2])
    observed_trigrams.add(trigram)

num_observed_trigrams = len(observed_trigrams)
print("Observed unique trigrams:", num_observed_trigrams)

total_possible_trigrams = V ** 3
zero_count_trigrams = total_possible_trigrams - num_observed_trigrams
print("Total possible trigrams from vocabulary:", total_possible_trigrams)
print("Zero count trigrams:", zero_count_trigrams)

Total words: 47
Unique words (V): 37
Observed unique trigrams: 45
Total possible trigrams from vocabulary: 50653
Zero count trigrams: 50608


In [2]:
print(words)

['jollof', 'rice', 'is', 'a', 'beloved', 'one-pot', 'rice', 'dish', 'from', 'west', 'africa', ',', 'characterized', 'by', 'its', 'vibrant', 'reddish-orange', 'color', 'and', 'rich', ',', 'often', 'slightly', 'spicy', ',', 'tomato-based', 'flavor', 'infused', 'with', 'onions', ',', 'peppers', ',', 'garlic', ',', 'ginger', ',', 'curry', 'powder', ',', 'and', 'thyme', ',', 'all', 'cooked', 'in', 'stock.']


In [3]:
import re

text = "jollof rice is a beloved one-pot rice dish from west africa , characterized by its vibrant reddish-orange color and rich , often slightly spicy , tomato-based flavor infused with onions , peppers , garlic , ginger , curry powder , and thyme , all cooked in stock."
# Let's split by spaces first, then if any word ends with a punctuation, separate it, or just use regex to split words and punctuation.
# Since commas already have spaces, regex finding words or punctuation:
words_fixed = re.findall(r'\w+(?:-\w+)*|[,.]', text)
print("Fixed words:", words_fixed)
print("Length:", len(words_fixed))

unique_words_fixed = set(words_fixed)
V_fixed = len(unique_words_fixed)
print("Unique words (V):", V_fixed)

observed_trigrams_fixed = set()
for i in range(len(words_fixed) - 2):
    trigram = (words_fixed[i], words_fixed[i+1], words_fixed[i+2])
    observed_trigrams_fixed.add(trigram)

num_observed_trigrams_fixed = len(observed_trigrams_fixed)
print("Observed unique trigrams:", num_observed_trigrams_fixed)

total_possible_trigrams_fixed = V_fixed ** 3
zero_count_trigrams_fixed = total_possible_trigrams_fixed - num_observed_trigrams_fixed
print("Zero count trigrams fixed:", zero_count_trigrams_fixed)

Fixed words: ['jollof', 'rice', 'is', 'a', 'beloved', 'one-pot', 'rice', 'dish', 'from', 'west', 'africa', ',', 'characterized', 'by', 'its', 'vibrant', 'reddish-orange', 'color', 'and', 'rich', ',', 'often', 'slightly', 'spicy', ',', 'tomato-based', 'flavor', 'infused', 'with', 'onions', ',', 'peppers', ',', 'garlic', ',', 'ginger', ',', 'curry', 'powder', ',', 'and', 'thyme', ',', 'all', 'cooked', 'in', 'stock', '.']
Length: 48
Unique words (V): 38
Observed unique trigrams: 46
Zero count trigrams fixed: 54826


In [4]:
raw_text = """jollof rice is a beloved one-pot rice dish from west africa , characterized by its vibrant reddish-orange color and rich , often slightly spicy , tomato-based flavor infused with onions , peppers , garlic , ginger , curry powder , and thyme , all cooked in stock."""
print(repr(raw_text))

'jollof rice is a beloved one-pot rice dish from west africa , characterized by its vibrant reddish-orange color and rich , often slightly spicy , tomato-based flavor infused with onions , peppers , garlic , ginger , curry powder , and thyme , all cooked in stock.'


In [5]:
from collections import Counter

counts = Counter(words_fixed)
print(counts.most_common())

[(',', 9), ('rice', 2), ('and', 2), ('jollof', 1), ('is', 1), ('a', 1), ('beloved', 1), ('one-pot', 1), ('dish', 1), ('from', 1), ('west', 1), ('africa', 1), ('characterized', 1), ('by', 1), ('its', 1), ('vibrant', 1), ('reddish-orange', 1), ('color', 1), ('rich', 1), ('often', 1), ('slightly', 1), ('spicy', 1), ('tomato-based', 1), ('flavor', 1), ('infused', 1), ('with', 1), ('onions', 1), ('peppers', 1), ('garlic', 1), ('ginger', 1), ('curry', 1), ('powder', 1), ('thyme', 1), ('all', 1), ('cooked', 1), ('in', 1), ('stock', 1), ('.', 1)]
